# Quantization-Aware Training (QAT) — MobileNetV3 → QAI Hub

**Vấn đề:** PTQ (Post-Training Quantization) thuần túy gây accuracy drop Rank-1 ~17.7%.

**Giải pháp:** Fine-tune model đã KD-train với **fake INT8 quantization noise** (QAT).
Weights học được sẽ robust hơn → PTQ sau đó drop ít hơn đáng kể.

**Workflow:**
| Bước | Mô tả |
|------|-------|
| 1 | Load KD student checkpoint |
| 2 | Deepcopy backbone+embedding → `EmbeddingModel` |
| 3 | Áp dụng eager-mode QAT (fake-quant observers) |
| 4 | Fine-tune `QAT_EPOCHS` epoch (mặc định 6): `α·MagFace + β·KD_cosine(student, teacher)` |
| 5 | Extract float32 weights (filter `activation_post_process` + `weight_fake_quant` keys) |
| 6 | Export ONNX → QAI Hub PTQ → compile → so sánh accuracy |

**Input:** `best_model.pth` (KD checkpoint) + `teacher albedo checkpoint`  
**Cách dùng:** Sửa `CONFIGURATION` ở cell config, chạy từ trên xuống.

## 1. Setup

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

REPO_URL    = 'https://github.com/NguyenXuanBinh22/DATN.git'
REPO_BRANCH = 'convnext-v2-dev'
REPO_DIR    = '/content/FR_Photometric_Stereo'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull origin {REPO_BRANCH}')
    print('Repo đã tồn tại, đã pull latest.')

%cd {REPO_DIR}
print(f'Working dir: {os.getcwd()}')

!pip install -q albumentations==1.3.1 timm tabulate
!pip install qai-hub onnx onnxruntime onnxscript

print('Setup xong.')

## 2. Imports & Cấu hình

In [ ]:
%cd /content/FR_Photometric_Stereo
import warnings
warnings.filterwarnings('ignore')

import copy
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.quantization
import albumentations as A
import onnxruntime as ort
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from tabulate import tabulate

from going_modular.dataloader.multitask import create_concatv2_multitask_datafetcher, create_eval_loaders
from going_modular.model.MTLFaceRecognition import MTLFaceRecognition
from going_modular.model.FaceRecognitionMobileNetV3 import FaceRecognitionMobileNetV3
from going_modular.loss.WeightClassMagLoss import WeightClassMagLoss
from going_modular.utils.transforms import RandomResizedCropRect, GaussianNoise
from going_modular.utils.roc_auc_id import compute_id_auc_gallery_probe, compute_rank1_gallery_probe
from going_modular.utils.ModelCheckPoint import ModelCheckpoint

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ════════════════════════════════════════════════════════════
#  CẤU HÌNH — chỉnh sửa ở đây
# ════════════════════════════════════════════════════════════
from google.colab import userdata
import qai_hub as hub

DRIVE_DATASET_DIR = '/content/drive/MyDrive/Photometric_DB_Full/'

# Checkpoint KD đã train (best_model.pth)
STUDENT_CKPT = (
    '/content/drive/MyDrive/experiments/'
    '(1-15-100-200)new_lan2_KD_CrossModal_ConcatTeacher_to_MobileNetV3_Albedo/'
    'checkpoints/best_model.pth'
)

# Teacher single-modal (albedo ConvNeXt) — dùng để KD loss trong QAT
TEACHER_CKPT_1 = '/content/drive/MyDrive/Photometric_DB_Full/experiments/Single_ALBEDO_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth'
USE_KD_LOSS    = True   # False = chỉ dùng MagFace (nếu teacher ckpt không có)

STUDENT_MODAL_IDX = 0   # 0 = albedo, 1 = normalmap

OUTPUT_DIR       = '/content/drive/MyDrive/experiments/qat_mobilenetv3/'
TARGET_DEVICE    = 'Samsung Galaxy S24 (Family)'
INPUT_SHAPE      = (1, 3, 112, 112)

# QAT hyper-params
QAT_EPOCHS = 6        # QAT fine-tune chỉ cần vài epoch để observer hội tụ, không phải train lại từ đầu
QAT_LR     = 1e-5     # 10× nhỏ hơn LR gốc (1e-4) để tránh catastrophic forgetting
TASK_W     = 1.0
KD_W       = 1.0      # giảm từ 5.0 → 1.0: model đã được KD đầy đủ ở bước train gốc rồi,
                       # KD_W quá cao trong QAT kéo embedding lệch theo teacher quá mạnh,
                       # làm mất margin giữa class (quan sát thực tế: AUC -3.2% nhưng Rank-1 -16.3%)

# QAI Hub
QAI_HUB_TOKEN = userdata.get('QAI_HUB_TOKEN').strip()
os.environ['QAI_HUB_API_TOKEN'] = QAI_HUB_TOKEN

CONFIGURATION = {
    'dataset_dir': DRIVE_DATASET_DIR,
    'type':        'concat_v2',   # train loader trả [B, 2, 3, H, W]
    'image_size':  112,
    'batch_size':  16,
    'num_workers': 2,
    'backbone':    'mobilenetv3_large_100',
    'teacher_backbone': 'convnextv2_tiny',
    'use_sampler': True,
    'device':      device,
    'num_classes': None,
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
!qai-hub configure --api_token {QAI_HUB_TOKEN}
qai_device = hub.Device(TARGET_DEVICE)

print(f'Student ckpt  : {STUDENT_CKPT}')
print(f'Teacher ckpt  : {TEACHER_CKPT_1}')
print(f'Use KD loss   : {USE_KD_LOSS}')
print(f'QAT epochs    : {QAT_EPOCHS}  LR={QAT_LR}')
print(f'Target device : {TARGET_DEVICE}')

## 3. Data Loading

In [ ]:
dataset_dir = CONFIGURATION['dataset_dir']
train_csv   = os.path.join(dataset_dir, 'train_split.csv')
df_train    = pd.read_csv(train_csv)
CONFIGURATION['num_classes'] = int(df_train['id'].max() + 1)
print(f'num_classes: {CONFIGURATION["num_classes"]}  |  train samples: {len(df_train)}')

train_transform = A.Compose([
    RandomResizedCropRect(CONFIGURATION['image_size']),
    GaussianNoise(p=0.2),
], additional_targets={'image2': 'image'})

test_transform = A.Compose([
    A.Resize(CONFIGURATION['image_size'], CONFIGURATION['image_size']),
], additional_targets={'image2': 'image'})

# train_dl: batch [B, 2, 3, H, W] — student dùng X[:, STUDENT_MODAL_IDX]
train_dl, test_dl = create_concatv2_multitask_datafetcher(
    CONFIGURATION, train_transform, test_transform, 'train_split.csv', 'probe_split.csv'
)
print(f'Train batches: {len(train_dl)}  |  Test batches: {len(test_dl)}')

# Gallery-probe eval loader (single modal)
eval_conf = dict(CONFIGURATION)
eval_conf['type'] = ['albedo', 'normalmap'][STUDENT_MODAL_IDX]
gallery_dl, probe_dl = create_eval_loaders(
    eval_conf,
    A.Compose([A.Resize(CONFIGURATION['image_size'], CONFIGURATION['image_size'])])
)
print(f'Gallery: {len(gallery_dl)} batches  |  Probe: {len(probe_dl)} batches')

# Kiểm tra shape
X_s, y_s = next(iter(train_dl))
print(f'Batch shape: X={X_s.shape}, y={y_s.shape}')  # [B, 2, 3, H, W]

## 4. Load Models

### 4.1 Load KD Student Checkpoint

In [ ]:
if not os.path.exists(STUDENT_CKPT):
    raise FileNotFoundError(f'Không tìm thấy student checkpoint: {STUDENT_CKPT}')

student = FaceRecognitionMobileNetV3(
    num_classes=CONFIGURATION['num_classes'],
    backbone=CONFIGURATION['backbone'],
)
ckpt = torch.load(STUDENT_CKPT, map_location=device, weights_only=False)
student.load_state_dict(ckpt['model_state_dict'])
student.to(device)
print(f'Student loaded from epoch {ckpt.get("epoch", "?")}')
print(f'Params: {sum(p.numel() for p in student.parameters()):,}')

# Baseline accuracy trước QAT
student.eval()
pre_qat_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, student, device)
pre_qat_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, student, device)
print(f'\nBaseline (float32, trước QAT):')
print(f'  Cosine AUC : {pre_qat_auc["id_cosine"]:.4f}')
print(f'  Rank-1 Acc : {pre_qat_rank1:.4f}')

### 4.2 Load Single Teacher (albedo ConvNeXt — optional, cho KD loss)

In [ ]:
teacher_single = None

if USE_KD_LOSS:
    if not os.path.exists(TEACHER_CKPT_1):
        print(f'WARNING: Không tìm thấy teacher checkpoint: {TEACHER_CKPT_1}')
        print('Sẽ train QAT chỉ với MagFace loss.')
        USE_KD_LOSS = False
    else:
        t_ckpt = torch.load(TEACHER_CKPT_1, map_location=device, weights_only=False)
        t_sd   = {k.replace('module.', ''): v for k, v in t_ckpt['model_state_dict'].items()}

        # Infer num_classes từ checkpoint — tránh mismatch với CONFIGURATION['num_classes']
        # id_head.maglinear.weight: shape [emb_dim, num_classes]
        teacher_num_classes = t_sd['id_head.maglinear.weight'].shape[1]
        print(f'Teacher num_classes (from ckpt): {teacher_num_classes}')

        teacher_single = MTLFaceRecognition(
            backbone=CONFIGURATION['teacher_backbone'],
            num_classes=teacher_num_classes,   # dùng num_classes của teacher, không phải student
        )
        teacher_single.load_state_dict(t_sd, strict=False)
        teacher_single.to(device)
        teacher_single.eval()
        for p in teacher_single.parameters():
            p.requires_grad = False
        print(f'Teacher loaded: {TEACHER_CKPT_1}')
        print(f'Teacher params: {sum(p.numel() for p in teacher_single.parameters()):,} (frozen)')

        # Smoke test: teacher.get_result(X_single) → (emb_512, ...)
        with torch.no_grad():
            _x  = torch.randn(2, 3, 112, 112).to(device)
            _te = teacher_single.get_result(_x)[0]
            print(f'Teacher embedding shape: {_te.shape}')  # [2, 512]
else:
    print('USE_KD_LOSS=False — QAT chỉ dùng MagFace loss.')

## 5. QAT Preparation

**Chiến lược:**
- **Eager mode** (không dùng FX) để tránh Conv-BN fusion làm thay đổi state_dict keys
- Áp dụng QAT vào `EmbeddingModel` (backbone + embedding, deep copy)
- MagFace head giữ nguyên từ checkpoint gốc
- Sau khi train: filter `activation_post_process` keys → load vào clean model → export ONNX

In [ ]:
class EmbeddingModel(nn.Module):
    """
    Backbone + embedding — không có L2 normalize, không có MagLinear.
    Output: raw 512-D vector (chưa normalize).

    Đây là phần duy nhất được áp QAT vì:
    - MagLinear chỉ dùng khi train (bỏ khi export)
    - L2 normalize là op đơn giản, không cần quantize
    """
    def __init__(self, backbone, embedding):
        super().__init__()
        self.backbone  = backbone
        self.embedding = embedding

    def forward(self, x):
        return self.embedding(self.backbone(x))   # [B, 512] raw

    def get_embedding(self, x):
        """Cho compute_id_auc_gallery_probe compatibility."""
        with torch.no_grad():
            emb = self.forward(x)
        return F.normalize(emb, p=2, dim=1)


# Deepcopy để QAT không làm ô nhiễm student gốc
_backbone_copy  = copy.deepcopy(student.backbone)
_embedding_copy = copy.deepcopy(student.embedding)

embed_qat = EmbeddingModel(_backbone_copy, _embedding_copy).to(device)
print(f'EmbeddingModel params: {sum(p.numel() for p in embed_qat.parameters()):,}')

# Smoke test
with torch.no_grad():
    _dummy = torch.randn(2, 3, 112, 112).to(device)
    _out   = embed_qat(_dummy)
    print(f'Output shape: {_out.shape}')  # [2, 512]

In [ ]:
# ── Áp dụng QAT (eager mode) ─────────────────────────────────────────────────
# qnnpack: asymmetric activation (uint8), per-channel weight (int8) — gần nhất với Snapdragon
embed_qat.train()
embed_qat.qconfig = torch.quantization.get_default_qat_qconfig('qnnpack')
torch.quantization.prepare_qat(embed_qat, inplace=True)

# Kiểm tra xem activation_post_process đã được chèn vào chưa
post_process_count = sum(
    1 for name, _ in embed_qat.named_modules()
    if 'activation_post_process' in name
)
print(f'Fake-quant observer nodes: {post_process_count}')

# Tổng params sau khi thêm observers (thêm scale/zero_point, không thêm nhiều)
print(f'QAT model params: {sum(p.numel() for p in embed_qat.parameters()):,}')

# MagFace loss — dùng student.maglinear gốc (không deepcopy, cần cập nhật cùng backbone)
maglinear = student.maglinear  # tham chiếu thực, cập nhật cùng optimizer
criterion_task = WeightClassMagLoss(train_csv)
print('QAT preparation xong.')

## 6. QAT Fine-Tuning

**Lịch trình observer/BN (xem `cell-run-qat`):**
- Observers cập nhật bình thường tới epoch `QAT_EPOCHS - 1`, sau đó freeze (giữ nhiều epoch để scale/zero_point hội tụ ổn định)
- BN running stats chỉ freeze ở epoch cuối cùng (`QAT_EPOCHS`) — freeze sớm hơn sẽ khiến BN stats fit theo input đã nhiễu fake-quant, gây mismatch lớn khi eval pure float32 (không fake-quant) ở bước extract weights

In [ ]:
def train_epoch_qat(
    train_dl, embed_qat, maglinear, criterion_task,
    teacher_single, optimizer, device, student_modal_idx,
    task_w, kd_w, use_kd
):
    embed_qat.train()
    maglinear.train()

    total_loss = total_task = total_kd = 0.0
    for X, y in train_dl:
        X, y = X.to(device), y.to(device)
        id_labels = y[:, 0]

        X_student = X[:, student_modal_idx]  # [B, 3, H, W]

        # QAT forward: embedding với fake-quant noise
        emb = embed_qat(X_student)            # [B, 512] raw

        # Task loss — WeightClassMagLoss.forward(logits_tuple, target, x_norm)
        logits, norm = maglinear(emb)
        l_task = criterion_task(logits, id_labels, norm)
        loss   = task_w * l_task

        # KD loss: cosine giữa student (512) vs single_teacher (512)
        if use_kd and teacher_single is not None:
            with torch.no_grad():
                t_emb = teacher_single.get_result(X_student)[0]   # [B, 512]
            s_n = F.normalize(emb,   p=2, dim=1)
            t_n = F.normalize(t_emb, p=2, dim=1)
            l_kd = (1.0 - F.cosine_similarity(s_n, t_n, dim=1)).mean()
            loss = loss + kd_w * l_kd
            total_kd += l_kd.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_task += l_task.item()

    n = len(train_dl)
    return total_loss / n, total_task / n, total_kd / n


print('train_epoch_qat defined.')

In [ ]:
optimizer = Adam(
    list(embed_qat.parameters()) + list(maglinear.parameters()),
    lr=QAT_LR,
)
scheduler = CosineAnnealingLR(optimizer, T_max=QAT_EPOCHS, eta_min=1e-7)

best_auc  = 0.0
best_sd   = None
ckpt_path = os.path.join(OUTPUT_DIR, 'qat_best_embed.pth')

# Freeze observer muộn (gần cuối) để observer có nhiều epoch hội tụ scale/zero_point hơn;
# freeze BN running-stats chỉ ở epoch cuối cùng để giảm mismatch giữa train-time
# (fake-quant noise ON) và eval-time pure-float (fake-quant OFF) — BN running stats được
# fit trên input đã nhiễu fake-quant, freeze quá sớm sẽ làm pure-float inference lệch nhiều hơn.
OBSERVER_FREEZE_EPOCH = max(QAT_EPOCHS - 1, 1)
BN_FREEZE_EPOCH       = QAT_EPOCHS

print(f'QAT fine-tuning: {QAT_EPOCHS} epochs | LR={QAT_LR} | device={device}')
print(f'Loss: task_w={TASK_W}  kd_w={KD_W if USE_KD_LOSS else 0} (KD={USE_KD_LOSS})')
print(f'Observer freeze @ epoch {OBSERVER_FREEZE_EPOCH}  |  BN freeze @ epoch {BN_FREEZE_EPOCH}')
print('-' * 60)

for epoch in range(1, QAT_EPOCHS + 1):

    # ── Observer schedule ─────────────────────────────────────
    if epoch == OBSERVER_FREEZE_EPOCH:
        # Freeze observers: scale/zero_point không đổi nữa
        embed_qat.apply(torch.ao.quantization.disable_observer)
        print(f'  [Ep {epoch}] Observers frozen.')

    if epoch == BN_FREEZE_EPOCH:
        # Freeze BN running stats (chỉ ở epoch cuối, giảm train/eval mismatch)
        embed_qat.apply(torch.nn.intrinsic.qat.freeze_bn_stats)
        print(f'  [Ep {epoch}] BN stats frozen.')

    train_loss, train_task, train_kd = train_epoch_qat(
        train_dl, embed_qat, maglinear, criterion_task,
        teacher_single, optimizer, device, STUDENT_MODAL_IDX,
        TASK_W, KD_W, USE_KD_LOSS,
    )
    scheduler.step()

    # Eval: dùng embed_qat.eval() qua get_embedding wrapper
    embed_qat.eval()
    test_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, embed_qat, device)
    test_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, embed_qat, device)
    embed_qat.train()

    auc_cos = test_auc['id_cosine']

    print(f'Ep {epoch:02d}/{QAT_EPOCHS} | '
          f'loss={train_loss:.4f}  task={train_task:.4f}  kd={train_kd:.4f} | '
          f'AUC={auc_cos:.4f}  Rank1={test_rank1:.4f}')

    if auc_cos > best_auc:
        best_auc = auc_cos
        best_sd  = copy.deepcopy(embed_qat.state_dict())
        torch.save({'embed_state': best_sd, 'epoch': epoch, 'auc': best_auc}, ckpt_path)
        print(f'  → Best model saved (AUC={best_auc:.4f})')

print(f'\nQAT done. Best cosine AUC: {best_auc:.4f}')

## 7. Extract Float32 Weights & Export ONNX

Filter `activation_post_process` keys khỏi QAT state dict → load vào model sạch → export ONNX bình thường.
Model sạch này giữ nguyên float32 nhưng weights đã được train để robust với INT8 quantization noise.

In [ ]:
# Load best QAT checkpoint
best_qat_ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
embed_qat.load_state_dict(best_qat_ckpt['embed_state'])
embed_qat.eval()
print(f'Loaded best QAT embed (epoch {best_qat_ckpt["epoch"]}, AUC={best_qat_ckpt["auc"]:.4f})')

# ── Filter QAT-specific keys ──────────────────────────────────────────────────
# prepare_qat thêm 2 loại nodes vào state dict:
#   1. activation_post_process  — observers cho activation quantization
#   2. weight_fake_quant        — fake quant nodes cho weight quantization
# Cả 2 đều phải bỏ để lấy float32 weights thuần

QAT_KEYS = ('activation_post_process', 'weight_fake_quant')

qat_sd   = embed_qat.state_dict()
clean_sd = {k: v for k, v in qat_sd.items() if not any(s in k for s in QAT_KEYS)}

removed = len(qat_sd) - len(clean_sd)
print(f'State dict keys: {len(qat_sd)} (QAT) → {len(clean_sd)} (clean), removed {removed} observer keys')

# ── Load vào fresh student model ─────────────────────────────────────────────
student_qat = FaceRecognitionMobileNetV3(
    num_classes=CONFIGURATION['num_classes'],
    backbone=CONFIGURATION['backbone'],
)
# Load KD weights trước (để maglinear có weights)
ckpt_orig = torch.load(STUDENT_CKPT, map_location='cpu', weights_only=False)
student_qat.load_state_dict(ckpt_orig['model_state_dict'])

# Override backbone + embedding với QAT-trained weights (strict=False vì không có maglinear trong clean_sd)
missing, unexpected = student_qat.load_state_dict(clean_sd, strict=False)
print(f'Missing keys (expected — maglinear): {len(missing)}')
print(f'Unexpected keys: {len(unexpected)}')
assert len(unexpected) == 0, f'Có unexpected keys: {unexpected[:5]}'

student_qat.eval()

# Verify: accuracy student_qat (float32, QAT-trained weights)
post_qat_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, student_qat, device)
post_qat_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, student_qat, device)

print('\n--- Float32 accuracy: trước và sau QAT ---')
rows = [
    ['Cosine AUC',  f"{pre_qat_auc['id_cosine']:.4f}",  f"{post_qat_auc['id_cosine']:.4f}",
     f"{post_qat_auc['id_cosine'] - pre_qat_auc['id_cosine']:+.4f}"],
    ['Rank-1 Acc',  f'{pre_qat_rank1:.4f}',               f'{post_qat_rank1:.4f}',
     f'{post_qat_rank1 - pre_qat_rank1:+.4f}'],
]
print(tabulate(rows, headers=['Metric', 'Before QAT', 'After QAT', 'Δ'], tablefmt='fancy_grid'))
print('(Nếu Δ âm nhỏ thì OK — fine-tuning với noise đánh đổi 1 chút float32 acc để được INT8 acc tốt hơn)')

In [ ]:
class InferenceWrapper(nn.Module):
    """Backbone + embedding + L2 normalize — không có MagLinear."""
    def __init__(self, model):
        super().__init__()
        self.backbone  = model.backbone
        self.embedding = model.embedding

    def forward(self, x):
        return F.normalize(self.embedding(self.backbone(x)), p=2, dim=1)


inference_model = InferenceWrapper(student_qat).eval().cpu()
dummy_input     = torch.randn(1, 3, 112, 112)

ONNX_PATH = os.path.join(OUTPUT_DIR, 'qat_mobilenetv3_fr.onnx')

torch.onnx.export(
    inference_model,
    dummy_input,
    ONNX_PATH,
    input_names=['input'],
    output_names=['embedding'],
    dynamic_axes={'input': {0: 'batch'}, 'embedding': {0: 'batch'}},
    opset_version=17,
)
print(f'Exported ONNX: {ONNX_PATH}')
print(f'File size    : {os.path.getsize(ONNX_PATH) / 1024 / 1024:.1f} MB')

## 8. QAI Hub — PTQ trên QAT-trained Float32 ONNX

Flow giống notebook quantization gốc, nhưng bây giờ float32 model đã robust hơn với quantization noise.

In [ ]:
import onnx

print('Merging external weights into single ONNX file...')
model_proto    = onnx.load(ONNX_PATH, load_external_data=True)
ONNX_MERGED    = ONNX_PATH.replace('.onnx', '_merged.onnx')
onnx.save_model(model_proto, ONNX_MERGED, save_as_external_data=False)
print(f'Merged ONNX : {ONNX_MERGED}')
print(f'Size        : {os.path.getsize(ONNX_MERGED) / 1024 / 1024:.1f} MB')

In [ ]:
# ── Bước 1: compile ONNX → optimized ONNX ───────────────────────────────────
print('Compiling QAT ONNX → optimized ONNX...')
compile_onnx_job = hub.submit_compile_job(
    model=ONNX_MERGED,
    device=qai_device,
    input_specs=dict(input=INPUT_SHAPE),
    options='--target_runtime onnx',
)
assert isinstance(compile_onnx_job, hub.CompileJob)
opt_onnx_model = compile_onnx_job.get_target_model()
print(f'Optimized ONNX model ID: {opt_onnx_model.model_id}')

In [ ]:
import albumentations as A
import numpy as np

NUM_CALIBRATION_SAMPLES = 200
STUDENT_MODALITY        = ['albedo', 'normalmap'][STUDENT_MODAL_IDX]
file_map = {'albedo': 'albedo_map_new_crop.exr.npy', 'normalmap': 'normal_map_new_crop.exr.npy'}
file_suffix   = file_map[STUDENT_MODALITY]
infer_transform = A.Compose([A.Resize(112, 112)])

probe_csv = os.path.join(dataset_dir, 'probe_split.csv')
df_probe  = pd.read_csv(probe_csv)
df_calib  = df_probe.sample(n=min(NUM_CALIBRATION_SAMPLES, len(df_probe)), random_state=42)

sample_inputs = []
for _, row in df_calib.iterrows():
    npy_path = os.path.join(dataset_dir, str(row['id']), str(row['session']), file_suffix)
    try:
        img = np.load(npy_path)
        if img.ndim == 3 and img.shape[0] == 3:
            img = img.transpose(1, 2, 0)
        img = infer_transform(image=img.astype(np.float32))['image']
        img = np.expand_dims(np.transpose(img, (2, 0, 1)), 0)   # [1, 3, 112, 112]
        sample_inputs.append(img)
    except Exception:
        pass

calibration_data = dict(input=sample_inputs)
print(f'Calibration samples: {len(sample_inputs)} / {NUM_CALIBRATION_SAMPLES}')

In [ ]:
# ── Bước 2: PTQ (INT8 w8a8) ─────────────────────────────────────────────────
print('Submitting quantize job (INT8 w8a8) on QAT-trained model...')
quantize_job = hub.submit_quantize_job(
    model=opt_onnx_model,
    calibration_data=calibration_data,
    weights_dtype=hub.QuantizeDtype.INT8,
    activations_dtype=hub.QuantizeDtype.INT8,
)
assert isinstance(quantize_job, hub.QuantizeJob)
quantized_onnx_model = quantize_job.get_target_model()
print(f'Quantized ONNX model ID: {quantized_onnx_model.model_id}')

In [ ]:
# ── Bước 3: compile quantized ONNX → TFLite INT8 ───────────────────────────
print('Compiling quantized ONNX → TFLite INT8...')
compile_quant_job = hub.submit_compile_job(
    model=quantized_onnx_model,
    device=qai_device,
    options='--target_runtime tflite --quantize_io',
)
assert isinstance(compile_quant_job, hub.CompileJob)
quant_tflite_model = compile_quant_job.get_target_model()
print(f'Quantized TFLite model ID: {quant_tflite_model.model_id}')

In [ ]:
# ── Profile: latency của QAT INT8 model ─────────────────────────────────────
print('Profiling QAT INT8 TFLite...')
profile_job = hub.submit_profile_job(model=quant_tflite_model, device=qai_device)
assert isinstance(profile_job, hub.ProfileJob)
profile_data = profile_job.download_profile()
summary      = profile_data['execution_summary']

print(tabulate(
    [['Model',          'MobileNetV3 QAT-INT8 TFLite'],
     ['Device',         TARGET_DEVICE],
     ['Inference time', f"{summary.get('estimated_inference_time', '?')} ms"],
     ['Peak memory',    f"{summary.get('peak_memory_bytes', '?')} bytes"]],
    tablefmt='fancy_grid'
))

## 9. Đánh giá Accuracy — Float32 vs QAT-INT8

In [ ]:
import zipfile
import glob as _glob
import shutil

class OnnxModelWrapper(nn.Module):
    def __init__(self, onnx_path: str):
        super().__init__()
        self.session    = ort.InferenceSession(
            onnx_path,
            providers=['CUDAExecutionProvider', 'CPUExecutionProvider'],
        )
        self.input_name = self.session.get_inputs()[0].name

    def get_embedding(self, x: torch.Tensor) -> torch.Tensor:
        out = self.session.run(None, {self.input_name: x.cpu().numpy().astype(np.float32)})[0]
        return torch.from_numpy(out)

    def forward(self, x):
        return self.get_embedding(x)


# ── Download + extract quantized ONNX ───────────────────────────────────────
quant_onnx_base  = os.path.join(OUTPUT_DIR, 'qat_mobilenetv3_int8')
quant_onnx_zip   = quant_onnx_base + '.onnx.zip'
quant_onnx_local = quant_onnx_base + '.onnx'

if not os.path.exists(quant_onnx_local):
    quantized_onnx_model.download(quant_onnx_base)   # hub saves as <base>.onnx.zip

    # QUAN TRỌNG: extract vào thư mục TẠM riêng, KHÔNG extract trực tiếp vào OUTPUT_DIR.
    # OUTPUT_DIR đã có sẵn qat_mobilenetv3_fr.onnx + qat_mobilenetv3_fr_merged.onnx
    # (từ bước export trước) — nếu glob *.onnx trên cả OUTPUT_DIR có thể nhặt nhầm
    # 1 trong 2 file float32 đó thay vì file model.onnx mới giải nén.
    extract_dir = quant_onnx_base + '_extracted'
    os.makedirs(extract_dir, exist_ok=True)

    with zipfile.ZipFile(quant_onnx_zip, 'r') as z:
        members = z.namelist()
        print(f'Zip contents: {members}')
        z.extractall(extract_dir)

    found = _glob.glob(os.path.join(extract_dir, '**', '*.onnx'), recursive=True)
    if not found:
        raise FileNotFoundError(f'Không tìm thấy .onnx sau khi extract. Zip contents: {members}')

    src_onnx = found[0]
    src_dir  = os.path.dirname(src_onnx)

    # Move tất cả file trong src_dir (model.onnx + model.data) sang OUTPUT_DIR
    for fname in os.listdir(src_dir):
        src_path = os.path.join(src_dir, fname)
        dst_path = quant_onnx_local if fname.endswith('.onnx') else os.path.join(OUTPUT_DIR, fname)
        shutil.move(src_path, dst_path)

    shutil.rmtree(extract_dir, ignore_errors=True)

    if not os.path.exists(quant_onnx_local):
        raise FileNotFoundError(f'Extraction thành công nhưng file không tồn tại: {quant_onnx_local}')

    print(f'Downloaded & extracted: {quant_onnx_local}')
    print(f'model.data exists: {os.path.exists(os.path.join(OUTPUT_DIR, "model.data"))}')
else:
    print(f'Cached: {quant_onnx_local}')

float_size = os.path.getsize(ONNX_MERGED) / 1024 / 1024
quant_size = os.path.getsize(quant_onnx_local) / 1024 / 1024
print(f'Float32 size: {float_size:.2f} MB  |  Quantized size: {quant_size:.2f} MB')
if abs(float_size - quant_size) < 0.05:
    print('WARNING: 2 file size gần như giống nhau — kiểm tra lại xem có nhặt nhầm file không!')

# ── Eval ─────────────────────────────────────────────────────────────────────
eval_transform = A.Compose([A.Resize(112, 112)])
gallery_dl_eval, probe_dl_eval = create_eval_loaders(eval_conf, eval_transform)

print('Evaluating float32 (QAT-trained weights)...')
float_model = OnnxModelWrapper(ONNX_MERGED)
float_auc   = compute_id_auc_gallery_probe(gallery_dl_eval, probe_dl_eval, float_model, device)
float_rank1 = compute_rank1_gallery_probe(gallery_dl_eval, probe_dl_eval, float_model, device)

print('Evaluating QAT-INT8 (quantized ONNX)...')
quant_model = OnnxModelWrapper(quant_onnx_local)
quant_auc   = compute_id_auc_gallery_probe(gallery_dl_eval, probe_dl_eval, quant_model, device)
quant_rank1 = compute_rank1_gallery_probe(gallery_dl_eval, probe_dl_eval, quant_model, device)

rows = [
    ['Cosine AUC',
     f"{pre_qat_auc['id_cosine']:.4f}",
     f"{float_auc['id_cosine']:.4f}",
     f"{quant_auc['id_cosine']:.4f}",
     f"{float_auc['id_cosine'] - quant_auc['id_cosine']:+.4f}"],
    ['Rank-1 Acc',
     f'{pre_qat_rank1:.4f}',
     f'{float_rank1:.4f}',
     f'{quant_rank1:.4f}',
     f'{float_rank1 - quant_rank1:+.4f}'],
]
print('\n--- Accuracy Summary ---')
print(tabulate(rows,
               headers=['Metric', 'Float32 (no QAT)', 'Float32 (QAT)', 'QAT-INT8', 'Drop (QAT→INT8)'],
               tablefmt='fancy_grid'))

## 10. Download TFLite Model

In [ ]:
tflite_base = os.path.join(OUTPUT_DIR, 'qat_mobilenetv3_int8')
tflite_path = tflite_base + '.tflite'

if not os.path.exists(tflite_path):
    quant_tflite_model.download(tflite_base)   # hub appends .tflite (or .tflite.zip)
    zip_path = tflite_base + '.tflite.zip'
    if os.path.exists(zip_path) and not os.path.exists(tflite_path):
        with zipfile.ZipFile(zip_path, 'r') as z:
            name = next(n for n in z.namelist() if n.endswith('.tflite'))
            data = z.read(name)
        with open(tflite_path, 'wb') as f:
            f.write(data)

print(f'Downloaded: {tflite_path}')
print(f'File size : {os.path.getsize(tflite_path) / 1024:.1f} KB')

In [ ]:
model_ids = {
    'opt_onnx_model_id':      opt_onnx_model.model_id,
    'quantized_onnx_model_id': quantized_onnx_model.model_id,
    'quant_tflite_model_id':   quant_tflite_model.model_id,
    'target_device':           TARGET_DEVICE,
    'student_modality':        STUDENT_MODALITY,
    'qat_epochs':              QAT_EPOCHS,
    'qat_lr':                  QAT_LR,
    'use_kd_loss':             USE_KD_LOSS,
    'best_qat_auc':            float(best_auc),
}

ids_path = os.path.join(OUTPUT_DIR, 'qai_hub_qat_model_ids.json')
with open(ids_path, 'w') as f:
    json.dump(model_ids, f, indent=2)
print(f'Saved: {ids_path}')
print(json.dumps(model_ids, indent=2))

## 11. Reload từ ID đã lưu (nếu session bị ngắt)

In [ ]:
# ids_path = os.path.join(OUTPUT_DIR, 'qai_hub_qat_model_ids.json')
# with open(ids_path) as f:
#     saved_ids = json.load(f)

# opt_onnx_model      = hub.get_model(saved_ids['opt_onnx_model_id'])
# quantized_onnx_model = hub.get_model(saved_ids['quantized_onnx_model_id'])
# quant_tflite_model   = hub.get_model(saved_ids['quant_tflite_model_id'])
# print('Models reloaded.')